In [ ]:
##rag
import os
import torch
import faiss
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from google.colab import drive

drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"

llm_name = "microsoft/phi-2"
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

embedder = SentenceTransformer(embed_model_name)

folder_path = "/content/drive/MyDrive/knowledge_base_folder"

knowledge_base = []

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)

    if file_name.endswith(".csv"):
        df = pd.read_csv(file_path)
        if "text" in df.columns:
            knowledge_base.extend(df["text"].dropna().tolist())

    elif file_name.endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
            lines = [line.strip() for line in lines if line.strip()]
            knowledge_base.extend(lines)

doc_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True, show_progress_bar=True)

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)


def retrieve_context(query, top_k=3):
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    retrieved_docs = [knowledge_base[i] for i in indices[0]]
    return " ".join(retrieved_docs)


def generate_rag_response(context, question, options):
    query = context + " " + question
    retrieved_text = retrieve_context(query)

    prompt = f"""
Use the following retrieved information to answer the question in a fair and evidence-based way.

Retrieved Information:
{retrieved_text}

Context: {context}
Question: {question}
Option 1: {options[0]}
Option 2: {options[1]}
Option 3: {options[2]}

Respond only with the correct answer.
The correct answer is:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            temperature=0.0
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("The correct answer is:")[-1].strip()
    return answer